# Land Cover Engine

## Purpose

The Land Cover Engine retrieves land cover information for the selected Area of Interest (AOI) and generates standardized land cover products for Earth Intelligence analyses.

Version 1 identifies:

- Forest
- Grassland
- Cropland
- Built-up Areas
- Water
- Bare Ground
- Wetlands
- Shrubland
- Snow and Ice
- Mangroves
- Moss and Lichen

The resulting Land Cover Product provides surface composition information for downstream modules including environmental analysis, ecosystem assessment, infrastructure planning, and risk assessment.

# Import Libraries

## Purpose

Import the libraries required for land cover retrieval, raster processing, and statistical analysis.

In [1]:
from pathlib import Path
import json

import geopandas as gpd
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray

import planetary_computer
from pystac_client import Client
from odc.stac import load

import matplotlib.pyplot as plt

# Load Inputs

## Purpose

Load the input datasets required by the Land Cover Engine.

The Land Cover Engine uses:

- Area of Interest (AOI)
- Earth Intelligence Catalog
- Satellite Product

These datasets provide the spatial boundary, land cover dataset information, and satellite metadata required for land cover retrieval and analysis.

In [2]:
from pathlib import Path
import json
import geopandas as gpd

DATA_DIR = Path(
    "/Users/ShreyaJariwalaMain/_GeoAI_Notebook/Earth-Intelligence-System/data/outputs"
)

# Load Area of Interest
aoi_file = DATA_DIR / "aoi.geojson"

if not aoi_file.exists():
    raise FileNotFoundError(
        "AOI not found. Run Notebook 01 first."
    )

aoi_boundary = gpd.read_file(aoi_file)

# Load Earth Intelligence Catalog
catalog_file = DATA_DIR / "catalog.json"

if not catalog_file.exists():
    raise FileNotFoundError(
        "Catalog not found. Run Notebook 02 first."
    )

with open(catalog_file, "r", encoding="utf-8") as file:
    catalog = json.load(file)

# Load Satellite Product
satellite_product_file = DATA_DIR / "satellite_product.json"

if not satellite_product_file.exists():
    raise FileNotFoundError(
        "Satellite Product not found. Run Notebook 03 first."
    )

with open(satellite_product_file, "r", encoding="utf-8") as file:
    satellite_product = json.load(file)

print("Inputs loaded successfully.")

Inputs loaded successfully.


# Land Cover Dataset Selection

## Purpose

Select the preferred land cover dataset from the Earth Intelligence Catalog.

The selection is based on dataset metadata rather than hardcoded dataset names.

Selection Criteria

- Category = Land Cover
- Applicable = True
- Highest Priority

The selected dataset becomes the source for land cover retrieval.

In [3]:
land_cover_datasets = [

    dataset

    for dataset in catalog["datasets"]

    if dataset["category"] == "Land Cover"
    and dataset["applicable"]

]

if len(land_cover_datasets) == 0:

    raise ValueError(
        "No applicable land cover datasets found."
    )

priority_order = {
    "Primary": 1,
    "Secondary": 2
}

land_cover_datasets = sorted(

    land_cover_datasets,

    key=lambda dataset: priority_order.get(
        dataset["priority"],
        99
    )

)

selected_dataset = land_cover_datasets[0]

selected_dataset

{'id': 'esa_worldcover',
 'name': 'ESA WorldCover',
 'category': 'Land Cover',
 'provider': 'European Space Agency',
 'access_method': 'STAC',
 'stac_collection': 'esa-worldcover',
 'description': 'Global land cover classification map.',
 'coverage': 'Global',
 'spatial_resolution': '10 m',
 'temporal_resolution': 'Annual',
 'data_type': 'Raster',
 'applicable': True,
 'priority': 'Primary',
 'notes': 'Preferred land cover dataset.'}

# Load Land Cover Data

## Purpose

Retrieve the land cover dataset for the selected Area of Interest (AOI).

The Land Cover Engine searches the selected STAC collection, retrieves the appropriate land cover product, and clips it to the AOI.

The resulting raster forms the basis for land cover analysis and statistics.

In [5]:
from shapely.geometry import mapping

catalog_client = Client.open(

    "https://planetarycomputer.microsoft.com/api/stac/v1",

    modifier=planetary_computer.sign_inplace

)

search = catalog_client.search(

    collections=[selected_dataset["stac_collection"]],

    intersects=mapping(
        aoi_boundary.geometry.iloc[0]
    )

)

land_cover_items = list(search.items())

if len(land_cover_items) == 0:

    raise ValueError(
        "No land cover data found for the selected AOI."
    )

selected_land_cover = land_cover_items[0]

selected_land_cover

<Item id=ESA_WorldCover_10m_2021_v200_N18E072>

# Load Land Cover Raster

## Purpose

Load the selected land cover dataset into memory.

The raster is clipped to the Area of Interest and stored as an xarray Dataset for further analysis.

In [6]:
land_cover = load(

    [selected_land_cover],

    bands=["map"],

    geopolygon=aoi_boundary.geometry.iloc[0],

    chunks={}

)

land_cover

<xarray.Dataset> Size: 2MB
Dimensions:      (latitude: 2024, longitude: 1131, time: 1)
Coordinates:
  * latitude     (latitude) float64 16kB 19.06 19.06 19.06 ... 18.89 18.89 18.89
  * longitude    (longitude) float64 9kB 72.79 72.79 72.79 ... 72.89 72.89 72.89
  * time         (time) datetime64[us] 8B 2021-01-01
    spatial_ref  int32 4B 4326
Data variables:
    map          (time, latitude, longitude) uint8 2MB dask.array<chunksize=(1, 2024, 1131), meta=np.ndarray>

# Generate Land Cover Layers

## Purpose

Convert the retrieved land cover raster into standardized land cover layers.

The numeric land cover class codes are translated into human-readable land cover classes based on the ESA WorldCover classification scheme.

These standardized layers serve as the foundation for land cover statistics and downstream Earth Intelligence analyses.

In [7]:
land_cover_classes = {

    10: "Tree Cover",

    20: "Shrubland",

    30: "Grassland",

    40: "Cropland",

    50: "Built-up",

    60: "Bare / Sparse Vegetation",

    70: "Snow and Ice",

    80: "Permanent Water",

    90: "Herbaceous Wetland",

    95: "Mangroves",

    100: "Moss and Lichen"

}

land_cover_layer = land_cover["map"].squeeze()

land_cover_layer

<xarray.DataArray 'map' (latitude: 2024, longitude: 1131)> Size: 2MB
dask.array<getitem, shape=(2024, 1131), dtype=uint8, chunksize=(2024, 1131), chunktype=numpy.ndarray>
Coordinates:
  * latitude     (latitude) float64 16kB 19.06 19.06 19.06 ... 18.89 18.89 18.89
  * longitude    (longitude) float64 9kB 72.79 72.79 72.79 ... 72.89 72.89 72.89
    spatial_ref  int32 4B 4326
    time         datetime64[us] 8B 2021-01-01
Attributes:
    nodata:   0

# Land Cover Statistics

## Purpose

Summarize the land cover composition of the Area of Interest.

This analysis computes:

- Pixel count
- Area percentage
- Land cover class

The resulting statistics provide a quantitative description of the Earth's surface within the selected AOI.

In [8]:
import numpy as np
import pandas as pd

values = land_cover_layer.values

values = values[~np.isnan(values)]

unique_classes, pixel_counts = np.unique(

    values.astype(int),

    return_counts=True

)

land_cover_statistics = pd.DataFrame({

    "Class Code": unique_classes,

    "Land Cover": [

        land_cover_classes.get(
            code,
            "Unknown"
        )

        for code in unique_classes

    ],

    "Pixel Count": pixel_counts

})

land_cover_statistics["Percentage"] = (

    land_cover_statistics["Pixel Count"]

    /

    land_cover_statistics["Pixel Count"].sum()

) * 100

land_cover_statistics = land_cover_statistics.sort_values(

    "Percentage",

    ascending=False

).reset_index(drop=True)

land_cover_statistics["Percentage"] = (

    land_cover_statistics["Percentage"]

    .round(2)

)

land_cover_statistics

,Class Code,Land Cover,Pixel Count,Percentage
0,80,Permanent Water,1355995,59.24
1,50,Built-up,609582,26.63
2,10,Tree Cover,195742,8.55
3,95,Mangroves,41837,1.83
4,60,Bare / Sparse Vegetation,36381,1.59
5,40,Cropland,26628,1.16
6,30,Grassland,22902,1.00
7,90,Herbaceous Wetland,77,0.00


# Calculate Land Cover Area

## Purpose

Calculate the physical area occupied by each land cover class.

The area is derived from the raster pixel count and spatial resolution, providing a more interpretable measure of land cover distribution within the Area of Interest.

In [9]:
# ESA WorldCover resolution (10 m)
pixel_size = 10  # meters

pixel_area_m2 = pixel_size * pixel_size

pixel_area_km2 = pixel_area_m2 / 1_000_000

land_cover_statistics["Area (km²)"] = (

    land_cover_statistics["Pixel Count"]

    * pixel_area_km2

)

land_cover_statistics["Area (km²)"] = (

    land_cover_statistics["Area (km²)"]

    .round(3)

)

land_cover_statistics

,Class Code,Land Cover,Pixel Count,Percentage,Area (km²)
0,80,Permanent Water,1355995,59.24,135.600
1,50,Built-up,609582,26.63,60.958
2,10,Tree Cover,195742,8.55,19.574
3,95,Mangroves,41837,1.83,4.184
4,60,Bare / Sparse Vegetation,36381,1.59,3.638
5,40,Cropland,26628,1.16,2.663
6,30,Grassland,22902,1.00,2.290
7,90,Herbaceous Wetland,77,0.00,0.008


# Land Cover Product

## Purpose

Create and populate the standardized Land Cover Product.

The Land Cover Product summarizes the selected dataset, land cover statistics, and processing metadata generated by the Land Cover Engine.

It serves as the primary output of the Land Cover Engine for downstream Earth Intelligence modules.

In [11]:
from datetime import datetime

dominant_class = land_cover_statistics.iloc[0]

land_cover_product = {

    "metadata": {

        "engine": "Land Cover Engine",

        "version": "1.0",

        "created_at": datetime.now().strftime(
            "%Y-%m-%d %H:%M:%S"
        )

    },

    "dataset": {

        "id": selected_dataset["id"],

        "name": selected_dataset["name"],

        "provider": selected_dataset["provider"],

        "category": selected_dataset["category"],

        "access_method": selected_dataset["access_method"],

        "stac_collection": selected_dataset["stac_collection"]

    },

    "land_cover": {

    "classes": len(land_cover_statistics),

    "dominant_class": dominant_class["Land Cover"],

    "dominant_percentage": float(
        dominant_class["Percentage"]
    ),

    "summary": [

        {

            "class": row["Land Cover"],

            "area_km2": float(
                row["Area (km²)"]
            ),

            "percentage": float(
                row["Percentage"]
            )

        }

        for _, row in land_cover_statistics.iterrows()

    ]

},

    "processing": {

        "classification_system": "ESA WorldCover",

        "resolution": "10 m"

    }

}

land_cover_product

{'metadata': {'engine': 'Land Cover Engine',
  'version': '1.0',
  'created_at': '2026-07-15 22:20:07'},
 'dataset': {'id': 'esa_worldcover',
  'name': 'ESA WorldCover',
  'provider': 'European Space Agency',
  'category': 'Land Cover',
  'access_method': 'STAC',
  'stac_collection': 'esa-worldcover'},
 'land_cover': {'classes': 8,
  'dominant_class': 'Permanent Water',
  'dominant_percentage': 59.24,
  'summary': [{'class': 'Permanent Water',
    'area_km2': 135.6,
    'percentage': 59.24},
   {'class': 'Built-up', 'area_km2': 60.958, 'percentage': 26.63},
   {'class': 'Tree Cover', 'area_km2': 19.574, 'percentage': 8.55},
   {'class': 'Mangroves', 'area_km2': 4.184, 'percentage': 1.83},
   {'class': 'Bare / Sparse Vegetation',
    'area_km2': 3.638,
    'percentage': 1.59},
   {'class': 'Cropland', 'area_km2': 2.663, 'percentage': 1.16},
   {'class': 'Grassland', 'area_km2': 2.29, 'percentage': 1.0},
   {'class': 'Herbaceous Wetland', 'area_km2': 0.008, 'percentage': 0.0}]},
 'proces

# Export Land Cover Product

## Purpose

Export the outputs generated by the Land Cover Engine.

The Land Cover Product metadata is exported as a JSON file.

The land cover raster is exported as a NetCDF dataset for downstream Earth Intelligence modules.

In [12]:
from pathlib import Path
import json
import xarray as xr

OUTPUT_DIR = Path(
    "/Users/ShreyaJariwalaMain/_GeoAI_Notebook/Earth-Intelligence-System/data/outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Export Land Cover Product
land_cover_product_file = OUTPUT_DIR / "land_cover_product.json"

with open(
    land_cover_product_file,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        land_cover_product,
        file,
        indent=4,
        default=str
    )

# Export Land Cover Layers
land_cover_dataset = xr.Dataset({

    "land_cover": land_cover_layer

})

land_cover_dataset_file = OUTPUT_DIR / "land_cover_layers.nc"

land_cover_dataset.to_netcdf(
    land_cover_dataset_file
)

print(f"Land Cover Product : {land_cover_product_file.name}")
print(f"Land Cover Layers  : {land_cover_dataset_file.name}")

Land Cover Product : land_cover_product.json
Land Cover Layers  : land_cover_layers.nc


# Land Cover Engine Summary

## Purpose

Summarize the outputs generated by the Land Cover Engine.

This summary confirms that the land cover dataset was successfully retrieved, processed, and summarized for the selected Area of Interest.

In [13]:
summary = {

    "Dataset": land_cover_product["dataset"]["name"],

    "Land Cover Classes": land_cover_product["land_cover"]["classes"],

    "Dominant Class": land_cover_product["land_cover"]["dominant_class"],

    "Dominant Percentage": (
        f"{land_cover_product['land_cover']['dominant_percentage']} %"
    ),

    "Resolution": land_cover_product["processing"]["resolution"]

}

for key, value in summary.items():

    print(f"{key:<22}: {value}")

Dataset               : ESA WorldCover
Land Cover Classes    : 8
Dominant Class        : Permanent Water
Dominant Percentage   : 59.24 %
Resolution            : 10 m
